一个普通 PyTorch Tensor 可以抽象为：

![img2](./src/chapter02.png)

> 官方文档同样把普通 Tensor 描述为 Storage、dtype、shape、stride、offset 的组合；多个 Tensor 可以共享同一个 Storage。

# Tensor 创建

In [68]:
import torch
x = torch.zeros( (2, 3), dtype=torch.float32, device="cpu" )
x

tensor([[0., 0., 0.],
        [0., 0., 0.]])

- shape=[2,3]
- dtype=torch.float32
- device=cpu

## 常见创建 API

| API                  | 含义                        |
| -------------------- | ------------------------- |
| `torch.tensor(data)` | 根据已有 Python / NumPy 等数据创建 |
| `torch.zeros(shape)` | 全 0                       |
| `torch.ones(shape)`  | 全 1                       |
| `torch.empty(shape)` | 只分配内存，不初始化                |
| `torch.rand(shape)`  | `[0, 1)` 均匀随机             |
| `torch.randn(shape)` | 标准正态分布                    |
| `torch.arange(...)`  | 等步长序列                     |
| `torch.from_numpy()` | 从 NumPy `ndarray` 创建      |

> PyTorch 官方教程明确指出，empty() 只分配内存而不初始化，所以看到的数值可能只是该内存此前残留的内容。

当把一个已有的 Tensor a 传给 torch.tensor(a) 时，PyTorch 底层执行的是重新分配物理内存与剥离计算图上下文的双重操作，等价于 a.detach().clone()。

## 内存层面：数据物理深拷贝（Copy Data）

PyTorch 的 Tensor 架构分为两部分：

- Tensor 头部元信息：记录形状（shape）、步长（stride）、数据类型（dtype）、所在设备（device）等。

- 底层的连续数据块（Storage）：实际存储数字的物理内存指针。

当执行 b = torch.tensor(a) 时，PyTorch 不会复用 a 的 Storage，而是向操作系统（或显卡驱动）申请一块全新的、与 a 大小相同的连续内存，把数据完整拷贝进去。

In [69]:
a = torch.tensor([1.0, 2.0])
b = torch.tensor(a)  # 会触发 PyTorch 的 UserWarning

# 验证内存地址
print(a.data_ptr() == b.data_ptr())  # 输出: False（内存独立）

# 修改 b 完全不会波及 a
b[0] = 999.0
print(a)  # tensor([1., 2.])
print(b)  # tensor([999., 2.])

False
tensor([1., 2.])
tensor([999.,   2.])


C:\Users\13127\AppData\Local\Temp\ipykernel_16156\2362816174.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  b = torch.tensor(a)  # 会触发 PyTorch 的 UserWarning


> 对比切片与视图（View）：常规的 b = a[:] 或 b = a.view(-1) 只创建新头部，底层 data_ptr() 相同，修改一个会导致另一个改变；而 torch.tensor(a) 彻底杜绝了这种数据耦合。

## torch.zeros / ones

In [70]:
a=torch.zeros(2,3)
a

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [71]:
b=torch.ones(2,3)
b

tensor([[1., 1., 1.],
        [1., 1., 1.]])

## torch.empty

申请 2×3 元素所需的内存 但不初始化里面的数据

适合：确定随后马上会全部覆盖数据

不适合：创建后直接拿它进行数学计算

In [72]:
x=torch.empty(2,3)
x

tensor([[-5.9518e+20,  1.3424e-42,  2.3694e-38],
        [ 2.3694e-38,  2.3694e-38,  2.3694e-38]])

## torch.rand

$$[x_{ij}\sim U(0,1)]$$

即 [0,1) 均匀随机数。

In [73]:
torch.rand(2,3)

tensor([[0.3427, 0.2201, 0.8974],
        [0.3201, 0.2561, 0.8678]])

## torch.randn

$$[x_{ij}\sim N(0,1)]$$
$$[\mu=0,\qquad \sigma=1]$$

In [74]:
torch.randn(2,3)

tensor([[ 1.0882, -1.2939,  1.0461],
        [ 0.7870, -0.9785,  0.4050]])

## torch.arange

形式：torch.arange(start, end, step)

区间为：[start, end)

In [75]:
torch.arange(0,10,2)

tensor([0, 2, 4, 6, 8])

## torch.from_numpy：必须理解“共享内存”

In [76]:
import numpy as np
import torch
a = np.array([1, 2, 3])
x = torch.from_numpy(a)
x

tensor([1, 2, 3])

官方明确说明 torch.from_numpy() 返回的 Tensor 与 ndarray 共享相同内存。

## Tensor 元信息

### .size()
x.size()与：x.shape基本等价。

In [77]:
x=torch.randn(32,128,768)
print(x.shape)
print(x.size())

torch.Size([32, 128, 768])
torch.Size([32, 128, 768])


In [78]:
x.size(0)

32

### .ndim

矩阵维度

In [79]:
x.ndim

3

### .numel()

元素总数

$$memory=numel\times element_size$$

In [80]:
x.numel()

3145728

### .dtype

torch.float32

torch.float16

torch.bfloat16

torch.int64

torch.bool

In [81]:
x.dtype

torch.float32

### .device

In [82]:
x.device

device(type='cpu')

### Indexing 与 Slicing

In [83]:
x=torch.arange(12).reshape(3,4)
x

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

### integer indexing

In [84]:
x[1]

tensor([4, 5, 6, 7])

### basic slicing

In [85]:
x[:,1:3]

tensor([[ 1,  2],
        [ 5,  6],
        [ 9, 10]])

# Tensor 数学运算

## 逐元素运算 Elementwise

In [86]:
a=torch.tensor([1.,2.,3.])
b=torch.tensor([4.,5.,6.])

### 加减乘除

In [87]:
a+b

tensor([5., 7., 9.])

In [88]:
a-b

tensor([-3., -3., -3.])

In [89]:
a*b

tensor([ 4., 10., 18.])

In [90]:
a/b

tensor([0.2500, 0.4000, 0.5000])

## 常见 elementwise function

In [91]:
x=torch.tensor([1,2,3])
torch.exp(x)

tensor([ 2.7183,  7.3891, 20.0855])

In [92]:
torch.log(x)

tensor([0.0000, 0.6931, 1.0986])

In [93]:
torch.relu(x)

tensor([1, 2, 3])

In [94]:
torch.sigmoid(x)

tensor([0.7311, 0.8808, 0.9526])

### comparison

In [95]:
x=torch.tensor([-1,0,2])
x>0

tensor([False, False,  True])

In [96]:
x==0

tensor([False,  True, False])

In [97]:
x!=0

tensor([ True, False,  True])

In [98]:
x<=1

tensor([ True,  True, False])

## Reduction：归约

### .sum()

x.sum():所有元素被压成一个 scalar

In [99]:
x=torch.range(1,12).reshape(3,4)
x

C:\Users\13127\AppData\Local\Temp\ipykernel_16156\2737782017.py:1: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  x=torch.range(1,12).reshape(3,4)


tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])

### 指定 dim

x.sum(dim=1) 意味着：沿 dim=1 归约

In [100]:
x.sum(dim=1)

tensor([10., 26., 42.])

### .mean()

In [101]:
x.mean(dim=0)

tensor([5., 6., 7., 8.])

### .max()

values, indices = x.max(dim=1)

不仅可以得到最大值，还能得到位置。

In [102]:
val,ids=x.max(dim=0)
print(val,ids)

tensor([ 9., 10., 11., 12.]) tensor([2, 2, 2, 2])


### keepdim=True

默认：x.sum(dim=1)

[B,T,D] -> [B,D]

但是：x.sum(dim=1, keepdim=True)

得到：[B,1,D]

官方对 sum 的定义也是：keepdim=True 时，被归约维度保留，但其 size 变为 1。

In [103]:
x=torch.range(1,24).reshape(2,3,4)
x

C:\Users\13127\AppData\Local\Temp\ipykernel_16156\1514554451.py:1: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  x=torch.range(1,24).reshape(2,3,4)


tensor([[[ 1.,  2.,  3.,  4.],
         [ 5.,  6.,  7.,  8.],
         [ 9., 10., 11., 12.]],

        [[13., 14., 15., 16.],
         [17., 18., 19., 20.],
         [21., 22., 23., 24.]]])

In [104]:
mean=x.mean(dim=1,keepdim=True)
mean

tensor([[[ 5.,  6.,  7.,  8.]],

        [[17., 18., 19., 20.]]])

In [105]:
mean.shape

torch.Size([2, 1, 4])

## Matrix Multiplication

### 二维矩阵

$$(M,K)(K,N)\rightarrow(M,N)$$

### @ 与 torch.matmul

A @ B 与：torch.matmul(A, B)

In [106]:
a=torch.range(1,12).reshape(3,4)
b=torch.range(1,12).reshape(4,3)
a

C:\Users\13127\AppData\Local\Temp\ipykernel_16156\2475413523.py:1: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  a=torch.range(1,12).reshape(3,4)
C:\Users\13127\AppData\Local\Temp\ipykernel_16156\2475413523.py:2: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  b=torch.range(1,12).reshape(4,3)


tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])

In [107]:
b

tensor([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 7.,  8.,  9.],
        [10., 11., 12.]])

In [108]:
a@b

tensor([[ 70.,  80.,  90.],
        [158., 184., 210.],
        [246., 288., 330.]])

In [109]:
torch.matmul(a,b)

tensor([[ 70.,  80.,  90.],
        [158., 184., 210.],
        [246., 288., 330.]])

## cat 与 stack

In [110]:
a=torch.randn(2,3)
b=torch.randn(2,3)

In [111]:
a

tensor([[ 0.6005,  0.0943, -0.7339],
        [-0.7070, -0.4682,  0.3949]])

In [112]:
b

tensor([[ 0.9121,  1.1247,  0.3015],
        [ 0.5649, -1.3977,  1.2442]])

### cat

torch.cat([a, b], dim=0)

In [113]:
torch.cat([a,b],dim=0)

tensor([[ 0.6005,  0.0943, -0.7339],
        [-0.7070, -0.4682,  0.3949],
        [ 0.9121,  1.1247,  0.3015],
        [ 0.5649, -1.3977,  1.2442]])

### stack

torch.stack([a, b], dim=0)

因为 stack：创建一个新的维度

官方对此的定义非常直接：stack 沿一个新维度连接，而 cat 沿已有维度连接。

In [114]:
y=torch.stack([a,b],dim=0)

In [115]:
y.shape


torch.Size([2, 2, 3])

# Shape Manipulation

## unsqueeze

作用：增加一个 size=1 的轴

官方说明 unsqueeze 返回的新 Tensor 与原 Tensor 共享底层数据。

In [118]:
x=torch.tensor([1,2,3])
x.shape

torch.Size([3])

In [120]:
x.unsqueeze(0).shape

torch.Size([1, 3])

In [121]:
x.unsqueeze(1).shape

torch.Size([3, 1])

## squeeze

作用：删除 size=1 的维度

例如：[32,1,128]

执行：x.squeeze(1)

得到：[32,128]

squeeze 同样属于 view operation，共享 Storage。

In [124]:
y=x.unsqueeze(0)
y.shape

torch.Size([1, 3])

In [125]:
y.squeeze(0).shape

torch.Size([3])

# View 与内存布局

## view

假设：x = torch.arange(6)

Storage：[0,1,2,3,4,5]

执行：y = x.view(2,3)

逻辑上：[[0,1,2],[3,4,5]]

但是：Storage 并没有变。

view：不复制底层数据,只重新解释 shape

官方明确说明 view Tensor 与 base Tensor 共享相同底层数据。

因此：

x = torch.arange(6)

y = x.view(2, 3)

y[0,0] = 100

那么：x

也会变成：[100,1,2,3,4,5]

In [126]:
x=torch.range(1,6)
x.view(2,3)

C:\Users\13127\AppData\Local\Temp\ipykernel_16156\2383600446.py:1: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  x=torch.range(1,6)


tensor([[1., 2., 3.],
        [4., 5., 6.]])

## -1

x.view(2, -1)

表示：这一维你帮我自动推导

In [127]:
print(x)
x.view(2,-1)

tensor([1., 2., 3., 4., 5., 6.])


tensor([[1., 2., 3.],
        [4., 5., 6.]])

## flatten

常见思想：[B,C,H,W] -> flatten(1) -> [B,C×H×W]

不要：x.flatten()

不加思考地把 batch 维也展开。

官方 nn.Flatten 默认 start_dim=1 正是为了常见神经网络 batch 场景。

In [129]:
x=torch.rand(2,3,4,5)
print(x.shape)
x.flatten(start_dim=1)

torch.Size([2, 3, 4, 5])


tensor([[0.1800, 0.0359, 0.8756, 0.4612, 0.9253, 0.8376, 0.5010, 0.9136, 0.7316,
         0.7863, 0.1950, 0.0175, 0.9184, 0.7857, 0.8739, 0.5570, 0.7112, 0.0426,
         0.2239, 0.6177, 0.1886, 0.0629, 0.5925, 0.8946, 0.1559, 0.8132, 0.5400,
         0.4054, 0.9650, 0.8269, 0.5849, 0.0205, 0.3773, 0.8563, 0.8126, 0.2192,
         0.1044, 0.5454, 0.5446, 0.0849, 0.0586, 0.4314, 0.7545, 0.3243, 0.0998,
         0.5317, 0.2363, 0.0669, 0.2842, 0.5168, 0.4451, 0.3600, 0.2817, 0.9445,
         0.3076, 0.9014, 0.4133, 0.5701, 0.1441, 0.9449],
        [0.9260, 0.0952, 0.8582, 0.4145, 0.0794, 0.0398, 0.1018, 0.1130, 0.9609,
         0.6904, 0.0618, 0.5430, 0.8334, 0.0597, 0.4749, 0.0199, 0.4811, 0.1540,
         0.6099, 0.7885, 0.5453, 0.3826, 0.4887, 0.2769, 0.4644, 0.5370, 0.1455,
         0.7013, 0.6011, 0.1872, 0.6403, 0.2427, 0.6158, 0.7900, 0.6956, 0.2715,
         0.2003, 0.4994, 0.3073, 0.0807, 0.2514, 0.2262, 0.7749, 0.0725, 0.8401,
         0.0237, 0.7698, 0.5076, 0.7708, 0.1483, 0.

## transpose

x.shape = [2,3]

执行：y = x.transpose(0,1)

得到：[3,2]

本质：交换两个 axis

In [130]:
x=torch.rand([1,2,3])
x

tensor([[[0.4670, 0.4580, 0.6112],
         [0.4111, 0.7237, 0.3465]]])

In [131]:
x.transpose(0,1)

tensor([[[0.4670, 0.4580, 0.6112]],

        [[0.4111, 0.7237, 0.3465]]])

## .T

二维矩阵：x.T

可以理解为：x.transpose(0,1)

但对高维 Tensor 不建议依赖 .T 来表达“矩阵转置”。

## permute

transpose：交换两个维度

permute：任意重新排列所有维度

## contiguous

> Tensor 当前的逻辑排列顺序，与底层连续内存排列顺序兼容。

例如：

0 1 2
3 4 5

Storage：

0 1 2 3 4 5

通常就是 contiguous。

检查：x.is_contiguous()

In [132]:
x=torch.rand(2,3)
x.is_contiguous()

True

In [134]:
x=x.transpose(0,1)
x.is_contiguous()

False